# 07 — Features (eye-tracking)

Gera a matriz de features do modelo (uma linha por participante):

1. **Por emoção** (feliz / neutro / raiva) — dispersão do olhar, velocidade, comprimento do
   scanpath, pupila, AOI central, e contrastes entre emoções → `dataset_dados_emocoes.csv`.
2. **Avançadas** (`src/features_avancadas.py`) — primeira fixação, fixações (I-DT), transições
   entre AOIs, entropia, revisitas → `dataset_features_avancadas.csv`.
3. **BCEA** — elipse de dispersão bivariada, no lugar do par `GazeX_Var` + `GazeY_Var`.

Saída final: `data/processed/dataset_features_et.csv` (252 features), entrada do `04_ML_et`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from features_avancadas import computar_dataset_features_avancadas

DIR_INTERIM   = RAIZ / "data" / "interim"
DIR_PROCESSED = RAIZ / "data" / "processed"
GAZE_COM_EMOCOES   = DIR_PROCESSED / "gaze_com_emocoes.csv"
JANELAS_ON_VALIDOS = DIR_INTERIM / "janelas_on_validos.csv"
ARQ_AOI            = DIR_INTERIM / "aoi.csv"
DATASET_EMOCOES    = DIR_PROCESSED / "dataset_dados_emocoes.csv"
DATASET_AVANCADAS  = DIR_PROCESSED / "dataset_features_avancadas.csv"
DATASET_FINAL      = DIR_PROCESSED / "dataset_features_et.csv"
META = ("ID_Sujeito", "Grupo", "Target")

## 1. Features por emoção + contrastes entre emoções

In [2]:
#gerar emocoes

import pandas as pd
import numpy as np

# ── Carregar o gaze com emoções (já gerado anteriormente) ──────
df_gaze = pd.read_csv(GAZE_COM_EMOCOES)

# Remover linhas 'off' (pós-experimento e ruído de 1 frame)
df_valido = df_gaze[df_gaze['emocao'].isin(['feliz', 'neutro', 'raiva'])].copy()

# ── Parâmetros da AOI geral (mesmos do dataset_dados_geral) ───
cx = 1024 / 2
cy = 768 / 2
r  = 80

# ── Funções auxiliares (mesmas da analise.ipynb) ──────────────
def calcular_estatisticas(series, prefixo):
    if len(series) == 0:
        return {f'{prefixo}_Media': 0, f'{prefixo}_Mediana': 0,
                f'{prefixo}_Std': 0,   f'{prefixo}_Var': 0,
                f'{prefixo}_Min': 0,   f'{prefixo}_Max': 0}
    return {
        f'{prefixo}_Media':   series.mean(),
        f'{prefixo}_Mediana': series.median(),
        f'{prefixo}_Std':     series.std(),
        f'{prefixo}_Var':     series.var(),
        f'{prefixo}_Min':     series.min(),
        f'{prefixo}_Max':     series.max(),
    }

def calcular_visitas_aoi(serie_bool):
    """Duração média dos blocos contínuos dentro da AOI (em frames)."""
    if len(serie_bool) == 0:
        return 0
    mudancas = serie_bool.ne(serie_bool.shift()).cumsum()
    contagem  = serie_bool.groupby(mudancas).sum()
    visitas   = contagem[contagem > 0]
    return visitas.mean() if len(visitas) > 0 else 0

# ── Loop principal ─────────────────────────────────────────────
registros  = []
excluidos  = []

print("Gerando dataset_dados_emocoes.csv...")

for sujeito in sorted(df_valido['ID_Sujeito'].unique()):
    df_sub  = df_valido[df_valido['ID_Sujeito'] == sujeito].copy()
    row0    = df_gaze[df_gaze['ID_Sujeito'] == sujeito].iloc[0]
    arquivo = row0['Arquivo']
    grupo   = row0['Grupo']
    target  = 0 if 'CONTROLE' in grupo.upper() else 1

    # Pular sujeitos sem as 3 emoções (T21, T35, T37 têm problema de auditoria)
    if not all(e in df_sub['emocao'].values for e in ['feliz', 'neutro', 'raiva']):
        excluidos.append(sujeito)
        print(f"  Excluído {sujeito}: emoções presentes = {list(df_sub['emocao'].unique())}")
        continue

    registro = {'ID_Sujeito': arquivo, 'Grupo': grupo, 'Target': target}

    for emocao in ['feliz', 'neutro', 'raiva']:
        df_em = df_sub[df_sub['emocao'] == emocao].sort_values('Time_ms').copy()
        total = len(df_em)
        suf   = f'_{emocao.capitalize()}'

        # ── AOI geral (círculo central) ────────────────────────
        df_em['dist']   = ((df_em['Gaze_X'] - cx)**2 + (df_em['Gaze_Y'] - cy)**2)**0.5
        df_em['inside'] = df_em['dist'] <= r
        perc_aoi        = df_em['inside'].sum() / total * 100
        dur_visita      = calcular_visitas_aoi(df_em['inside'])

        # ── Velocidade e distância total ───────────────────────
        df_clean = df_em[(df_em['Gaze_X'] != 0) & (df_em['Gaze_Y'] != 0)].copy()
        if len(df_clean) >= 2:
            dx  = df_clean['Gaze_X'].diff()
            dy  = df_clean['Gaze_Y'].diff()
            dt  = df_clean['Time_ms'].diff().replace(0, np.nan)
            vel = (np.sqrt(dx**2 + dy**2) / dt * 1000).dropna()
            dist_total = np.sqrt(dx**2 + dy**2).sum()
        else:
            vel        = pd.Series(dtype=float)
            dist_total = 0

        # ── Pupilometria ───────────────────────────────────────
        pupila = ((df_em['Pupil_Left_mm'].replace(0, np.nan) +
                   df_em['Pupil_Right_mm'].replace(0, np.nan)) / 2).dropna()

        # ── Dispersão espacial ─────────────────────────────────
        gaze_x = df_clean['Gaze_X'] if len(df_clean) > 0 else pd.Series(dtype=float)
        gaze_y = df_clean['Gaze_Y'] if len(df_clean) > 0 else pd.Series(dtype=float)

        registro.update({
            f'Perc_AOI_Geral{suf}':              perc_aoi,
            f'Duracao_Media_Visita_Frames{suf}':  dur_visita,
            f'Distancia_Total{suf}':              dist_total,
            **{f'{k}{suf}': v for k, v in calcular_estatisticas(vel,    'Velocidade').items()},
            **{f'{k}{suf}': v for k, v in calcular_estatisticas(pupila, 'Pupila').items()},
            **{f'{k}{suf}': v for k, v in calcular_estatisticas(gaze_x, 'GazeX').items()},
            **{f'{k}{suf}': v for k, v in calcular_estatisticas(gaze_y, 'GazeY').items()},
        })

    registros.append(registro)

df_emocoes = pd.DataFrame(registros)

# ── Features de delta (mudança entre emoções) ─────────────────
# Captura como o padrão de olhar MUDA entre expressões emocionais
# — o marcador comportamental central do TEA
METRICAS_DELTA = [
    'Perc_AOI_Geral', 'Duracao_Media_Visita_Frames', 'Distancia_Total',
    'Velocidade_Media', 'Velocidade_Mediana', 'Velocidade_Std',
    'Pupila_Media', 'Pupila_Mediana', 'Pupila_Std',
    'GazeX_Media', 'GazeX_Std', 'GazeY_Media', 'GazeY_Std',
]

for m in METRICAS_DELTA:
    cf, cn, cr = f'{m}_Feliz', f'{m}_Neutro', f'{m}_Raiva'
    if all(c in df_emocoes.columns for c in [cf, cn, cr]):
        df_emocoes[f'{m}_Delta_FelizRaiva']  = df_emocoes[cf] - df_emocoes[cr]
        df_emocoes[f'{m}_Delta_NeutroRaiva'] = df_emocoes[cn] - df_emocoes[cr]
        df_emocoes[f'{m}_Delta_FelizNeutro'] = df_emocoes[cf] - df_emocoes[cn]

# ── Salvar ─────────────────────────────────────────────────────
df_emocoes.to_csv(DATASET_EMOCOES, index=False)

print(f"\n✓ dataset_dados_emocoes.csv salvo!")
print(f"  Sujeitos: {len(df_emocoes)} (TEA={int((df_emocoes['Target']==1).sum())} | Controle={int((df_emocoes['Target']==0).sum())})")
print(f"  Excluídos: {excluidos} (sem as 3 emoções — problema de auditoria)")
print(f"  Features: {len(df_emocoes.columns)} colunas")
print(f"\n  Colunas por categoria:")
print(f"    - Por emoção (Feliz/Neutro/Raiva): {len([c for c in df_emocoes.columns if any(e in c for e in ['_Feliz','_Neutro','_Raiva']) and 'Delta' not in c])}")
print(f"    - Delta entre emoções: {len([c for c in df_emocoes.columns if 'Delta' in c])}")

Gerando dataset_dados_emocoes.csv...



✓ dataset_dados_emocoes.csv salvo!
  Sujeitos: 41 (TEA=18 | Controle=23)
  Excluídos: [] (sem as 3 emoções — problema de auditoria)
  Features: 123 colunas

  Colunas por categoria:
    - Por emoção (Feliz/Neutro/Raiva): 81
    - Delta entre emoções: 39


## 2. Features avançadas (primeira fixação, scanpath, transições entre AOIs)

In [3]:
df_av = computar_dataset_features_avancadas(
    gaze_csv    = GAZE_COM_EMOCOES,
    janelas_csv = JANELAS_ON_VALIDOS,
    aoi_csv     = ARQ_AOI,
    verbose     = True,
)
df_av.to_csv(DATASET_AVANCADAS, index=False)
print(f"\nSalvo: {df_av.shape[0]} sujeitos × {df_av.shape[1]} colunas")

AOIs medianas carregadas: ['Olho Direito', 'Olho Esquerdo', 'Nariz', 'Boca']
  Olho Direito: x=[392,514] y=[283,379]
  Olho Esquerdo: x=[506,629] y=[281,382]
  Nariz: x=[454,577] y=[302,438]
  Boca: x=[428,595] y=[390,500]


  T12 ✓


  T15 ✓


  T16 ✓
  T17 ✓


  T18 ✓


...

  T57 ✓


  T58 ✓


  T59 ✓


  T60 ✓


  T61 ✓

✓ Features avançadas geradas!
  Sujeitos: 41 (TEA=18 | Controle=23)
  Excluídos: []
  Features: 138 colunas

Salvo: 41 sujeitos × 138 colunas


## 3. BCEA e matriz final

BCEA = 2π·σx·σy·√(1 − ρ²): dispersão 2D do olhar que incorpora a correlação X–Y. Substitui, em
cada emoção, o par `GazeX_Var` + `GazeY_Var` (que é só o quadrado do desvio-padrão já presente).

In [4]:
# --- BCEA (bivariate contour ellipse area) por sujeito × emoção ---
def bcea(gx, gy):
    m = (gx != 0) & (gy != 0)
    gx, gy = gx[m], gy[m]
    if len(gx) < 3:
        return 0.0
    sx, sy = gx.std(), gy.std()
    rho = np.corrcoef(gx, gy)[0, 1] if len(gx) > 1 else 0.0
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

df_emoc = pd.read_csv(DATASET_EMOCOES)
df_av   = pd.read_csv(DATASET_AVANCADAS)
df_gaze = pd.read_csv(GAZE_COM_EMOCOES)

EMOS = [("feliz", "Feliz"), ("neutro", "Neutro"), ("raiva", "Raiva")]
df_bcea = pd.DataFrame([
    {"ID_Sujeito": arq, **{f"BCEA_{suf}": bcea(g.loc[g["emocao"] == emo, "Gaze_X"].values,
                                               g.loc[g["emocao"] == emo, "Gaze_Y"].values)
                           for emo, suf in EMOS}}
    for arq, g in df_gaze.groupby("Arquivo")])

# troca GazeX_Var_<emo> + GazeY_Var_<emo> por BCEA_<emo>, mantendo a posição da coluna
cols_em, feitos = [], set()
for col in [x for x in df_emoc.columns if x not in META]:
    suf = next((s for _, s in EMOS if col in (f"GazeX_Var_{s}", f"GazeY_Var_{s}")), None)
    if suf is None:
        cols_em.append(col)
    elif suf not in feitos:
        cols_em.append(f"BCEA_{suf}"); feitos.add(suf)
cols_av = [x for x in df_av.columns if x not in META]

df_final = (df_emoc.merge(df_bcea, on="ID_Sujeito", how="left")
                   .merge(df_av[["ID_Sujeito"] + cols_av], on="ID_Sujeito", how="inner"))
df_final = df_final[list(META) + cols_em + cols_av]          # ordem dos sujeitos = ordem de ID
df_final.to_csv(DATASET_FINAL, index=False)
print(f"{DATASET_FINAL.name}: {len(df_final)} sujeitos "
      f"(TEA={int((df_final.Target == 1).sum())}, Controle={int((df_final.Target == 0).sum())}) "
      f"× {len(cols_em) + len(cols_av)} features  [emoções: {len(cols_em)}, avançadas: {len(cols_av)}]")

dataset_features_et.csv: 41 sujeitos (TEA=18, Controle=23) × 252 features  [emoções: 117, avançadas: 135]
